In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
df = pd.read_csv("lab_ticket_resolution.csv")
print(df.shape)
print(df.dtypes)
print(df.describe(include="all"))
print(df.isna().sum())

(240, 6)
ticket_id                    str
complexity_score           int64
lines_changed              int64
team_experience_years    float64
reopened                   int64
resolution_hours         float64
dtype: object
       ticket_id  complexity_score  lines_changed  team_experience_years  \
count        240        240.000000     240.000000             240.000000   
unique       240               NaN            NaN                    NaN   
top        T0001               NaN            NaN                    NaN   
freq           1               NaN            NaN                    NaN   
mean         NaN          5.491667      54.820833               6.105417   
std          NaN          2.979697      26.145017               3.391426   
min          NaN          1.000000       1.000000               0.500000   
25%          NaN          3.000000      33.000000               3.300000   
50%          NaN          5.500000      54.000000               5.800000   
75%          NaN   

In [2]:
X = df[["complexity_score", "team_experience_years"]]
y = df["resolution_hours"]

The features available at the intended prediction time are only complexity score and team experience years. I have excluded ticket_id because its just an identifier and doesn't add any information for the data we are using. I have also excluded the column 'reopened' because even though we have it in the data we have for now, it won't be available at prediction time and will only be available after the task is closed and later reopened. But we are talking about new tasks for now.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
models = {
    "Dummy (Mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Polynomial (Deg 2)": make_pipeline(
        PolynomialFeatures(degree=2), LinearRegression()
    ),
    "Polynomial (Deg 3)": make_pipeline(
        PolynomialFeatures(degree=3), LinearRegression()
    ),
}

results_list = []

for name, model in models.items():
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    results_list.append({"Model": name, "MAE": mae, "MSE": mse, "R² Score": r2})

df_comparison = pd.DataFrame(results_list)
print(df_comparison.to_string(index=False, float_format="%.4f"))

             Model    MAE      MSE  R² Score
      Dummy (Mean) 9.1140 112.3109   -0.0210
 Linear Regression 2.9060  12.7611    0.8840
Polynomial (Deg 2) 3.0202  13.9944    0.8728
Polynomial (Deg 3) 3.1398  15.5936    0.8582
